In [1]:
import os
import pennylane as qml
import tensorflow as tf
import pandas as pd
import numpy as np
from tensorflow.keras.layers import LeakyReLU

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

2026-02-03 15:25:33.575581: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-03 15:25:34.058380: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/lib64/openmpi/lib:/usr/local/cuda/lib64:/opt/intel/oneapi/lib/intel64:/opt/nvidia/cudaq/lib:/opt/postgresql/12.4/lib
2026-02-03 15:25:34.058496: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/lib64/openmpi/lib:/usr/local/cuda/lib64:/opt/intel/oneapi/lib/intel64:/opt/nvidia/cudaq/lib:/opt/postgresql/12.4/lib
2026-02-03 15:25:34.058503: W tensorflow/compiler/tf

In [2]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
  # Restrict access of TensorFlow to specific GPU
  try:
    tf.config.experimental.set_visible_devices(gpus[0], 'GPU')
  except RuntimeError as e:
    # Visible devices must be set at program startup
    print(e)

#tf.config.set_visible_devices([], 'GPU')

In [3]:
tf.config.experimental.get_visible_devices()

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'),
 PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [4]:
experiment_configs = [
    {"ansatz": "QNN", "lookback": 1},
    {"ansatz": "QNN", "lookback": 2},
    {"ansatz": "Wavelets", "lookback": 1},
    {"ansatz": "Wavelets", "lookback": 2},
]
prevision_window=[1,2,3,4,5,6]
max_layers = 4

# Importing Models

In [5]:
def qnode_circuit(inputs, weights):
    # weights: (n_layers,n_qubits,3)
    n_layers = len(weights)
    n_qubits = len(weights[0])
    
    ###############
    # Feature Map #
    ###############
    for idx in range(n_qubits):
        qml.Hadamard(wires=idx)
    qml.templates.AngleEmbedding(inputs, rotation='Y', wires=range(n_qubits))
    
    ##########
    # Ansatz #
    ##########
    for k in range(n_layers):
        # Variational Layer
        for i in range(len(weights[k])):
            qml.Rot(*weights[k][i],wires=i)

        # Entangling Layer
        for i in range(0, n_qubits-1): 
            qml.CNOT(wires=[i, i + 1])
        qml.CNOT(wires=[n_qubits-1, 0])
    
    ###############
    # Measurement #
    ###############
    return [qml.expval(qml.PauliZ(wires=i)) for i in range(n_qubits)]


In [6]:
def create_quantum_model(n_qubits, weights, prev):
    input_layer = tf.keras.layers.Input(shape=(n_qubits,))
    
    dev = qml.device("default.qubit", wires=n_qubits)
    qnn = qml.QNode(qnode_circuit, dev, interface="tf")
    q_layer = qml.qnn.KerasLayer(qnn, weights, output_dim=n_qubits)

    activation=tf.keras.layers.Activation(tf.keras.activations.relu)
    output_layer = tf.keras.layers.Dense(len(prev), LeakyReLU(alpha=0.01))

    model = tf.keras.models.Sequential([
        input_layer
        , q_layer
        , activation
        , output_layer])
    
    opt = tf.keras.optimizers.Adam(learning_rate = 0.001)
    
    model.compile(loss=['mse'], optimizer=opt, metrics=['mae'])
    
    return model

In [7]:
all_models = {}

#for config in experiment_configs:
    #ansatz = config["ansatz"]
    #lookback = config["lookback"]
ansatz = "QNN"
lookback = 1
subdir = f"Ansatz-{ansatz}-Lookback-{lookback}"

print(f"\n==============================")
print(f"Processando: Ansatz={ansatz}, Lookback={lookback}")
print(f"==============================")

if ansatz == "QNN":
    if lookback == 1:
        nqubits = 16
    else:
        nqubits = 32
else:
    if lookback == 1:
        nqubits = 86
    else:
        nqubits = 172

path_chk = os.path.abspath(os.path.join(os.getcwd(), 'checkpoint', subdir))
if not os.path.exists(path_chk):
    os.makedirs(path_chk)

for i in range(max_layers):
    weight_shapes = {"weights": (i+1,nqubits,3)}
    model = create_quantum_model(nqubits, weight_shapes, prevision_window)
    model.load_weights(os.path.join(path_chk, f'cp-{ansatz}-lookback-{lookback}-depth-{i+1}-0100.ckpt'))
    all_models[ansatz+"-"+str(lookback)+"-"+str(i+1)] = model



Processando: Ansatz=QNN, Lookback=1


# Data Loading

In [8]:
def load_table(path, prev, lookback):
    X = pd.read_csv(path)
    X.dropna(axis=0,how='any',inplace=True)
    
    # We remove all outliers from dataset
    X = X[X['EXT_PM_25'] < 1000] 

    if lookback > 1:
        for col in X.columns:
            X[col+'_Lookback'] = X.loc[:,col].shift(lookback-1)
        X = X.iloc[lookback-1:, :]

    # We copy the values in X to prepare the y dataset. The first row is removed from y 
    # since it does not have a previous value to serve as forecast
    y = X[:].drop(X.index[0])
    
    # We remove the last line in X since it doesn't have an equivalent y
    X = X.iloc[:-prev[-1],:]
    
    # We create the final y dataset by creating a new column with the predictions and 
    # removing the unnecessary information
    for i in prev:
        y[f'Prev {i} hour'] = y.loc[:,"EXT_PM_25"].shift(-(i-1))
    
    if prev[-1] == 1:
        y= y.iloc[:, -1:]
    else:
        y= y.iloc[:-(prev[-1]-1), -len(prev):]

    return X, y.values

In [9]:
print("\nLoading Datasets\n")
path_stoke = os.path.abspath(os.path.join(os.getcwd(), 'data', 'stoke'))
path_suther = os.path.abspath(os.path.join(os.getcwd(), 'data', 'suther'))

filename_train = "wavelets-lvl5-train.csv" if ansatz == "Wavelets" else "train.csv"
filename_test  = "wavelets-lvl5-test.csv"  if ansatz == "Wavelets" else "test.csv"

train_file_stoke  = os.path.join(path_stoke, filename_train)
train_file_suther = os.path.join(path_suther, filename_train)
test_file_stoke   = os.path.join(path_stoke, filename_test)
test_file_suther  = os.path.join(path_suther, filename_test)

print(f"importing data from {train_file_stoke}")
X_train_stoke, y_train_stoke = load_table(train_file_stoke, prevision_window, lookback)
print(f"importing data from {train_file_suther}")
X_train_suther, y_train_suther = load_table(train_file_suther, prevision_window, lookback)

X_all = pd.concat([X_train_stoke, X_train_suther], axis=0)
y_all = np.vstack((y_train_stoke,y_train_suther))

#X_all = X_train_stoke
#y_all = y_train_stoke


print(f"importing data from {test_file_stoke}")
X_test_stoke, y_test_stoke = load_table(test_file_stoke, prevision_window, lookback)
print(f"importing data from {test_file_suther}")
X_test_suther, y_test_suther = load_table(test_file_suther, prevision_window, lookback)

X_test = pd.concat([X_test_stoke, X_test_suther], axis=0)
y_test = np.vstack((y_test_stoke,y_test_suther))

#X_test = X_test_stoke
#y_test = y_test_stoke


plot_time = X_test['Time'].values


Loading Datasets

importing data from /home/otto.pires/dynex-poc/data/stoke/train.csv
importing data from /home/otto.pires/dynex-poc/data/suther/train.csv
importing data from /home/otto.pires/dynex-poc/data/stoke/test.csv
importing data from /home/otto.pires/dynex-poc/data/suther/test.csv


In [10]:
if lookback > 1:
    X_all = X_all.drop(["Time", "Month", "Time_Lookback", "Month_Lookback"], axis=1)
    X_test = X_test.drop(["Time", "Month", "Time_Lookback", "Month_Lookback"], axis=1)
else:
    X_all = X_all.drop(["Time", "Month"], axis=1)
    X_test = X_test.drop(["Time", "Month"], axis=1)

In [11]:
print(f"\nThere are {X_all.shape[1]} features and {X_all.shape[0]} instances in All Train set\n")
print(X_all.head())
print(f"\nThere are {X_test.shape[1]} features and {X_test.shape[0]} instances in All Test set\n")
print(X_test.head())


There are 16 features and 4352 instances in All Train set

        TEMP        HUM       PRESS  EXT_PM_25    CO_PRE    NO2_PRE  \
0  14.305167  46.894333  101.311333  12.333333  0.235167  14.397885   
1  13.923167  46.929833  101.320000  15.016667  0.226035  16.237973   
2  13.794167  47.107667  101.349667  17.533333  0.219977  16.949071   
3  13.798500  47.212667  101.387333  18.966667  0.218026  16.976633   
4  13.862167  47.418500  101.429167  13.700000  0.215322  16.410233   

      O3_PRE  SIN HOUR  COS HOUR   SIN DAY  COS DAY  SIN MONTH     COS MONTH  \
0  21.885943  0.000000  1.000000  0.201299  0.97953        1.0  6.123234e-17   
1  18.874601  0.258819  0.965926  0.201299  0.97953        1.0  6.123234e-17   
2  18.829282  0.500000  0.866025  0.201299  0.97953        1.0  6.123234e-17   
3  19.074056  0.707107  0.707107  0.201299  0.97953        1.0  6.123234e-17   
4  19.427294  0.866025  0.500000  0.201299  0.97953        1.0  6.123234e-17   

         LAT      LONG  ALT  
0 

In [12]:
print("\nScaling Data\n")
scaler_x = MinMaxScaler(feature_range=(0, 1))
Xs_all  = scaler_x.fit_transform(X_all)
Xs_test = scaler_x.transform(X_test)


Scaling Data



In [13]:
all_preds = {}
for key in all_models:
    all_preds[key] = all_models[key].predict(Xs_all,verbose=1)

136/136 [==============================] - 132s 973ms/step


# Statistical Analysis

In [14]:
from scipy import stats
path_stat = os.path.abspath(os.path.join(os.getcwd(), 'statistics', subdir))
if not os.path.exists(path_stat):
    os.makedirs(path_stat)

In [15]:
def verify_distribution_wilcoxtest(data1, data2, p_H0):
    stat, p = stats.wilcoxon(data1, data2)
    print('Statistics=%.3f, p=%.3f' % (stat, p))
    if p > p_H0:
        print('Same distribution (fail to reject H0)')
    else:
        print('Different distribution (reject H0)')
    return stat, p

In [16]:
def verify_distribution_shapiro(data, p_H0=0.05):
    stat, p = stats.shapiro(data)
    print(f'Statistics={stat}, p={p}')
    if p > p_H0:
        print('Probably normal distribution (fail to reject H0)')
    else:
        print('Does not follow normal distribution (reject H0)')
    return stat, p

In [17]:
def verify_distribution_kruskal(*groups, p_H0=0.05):
    stat, p = stats.kruskal(*groups)
    print(f'Statistics={stat}, p={p}')
    for hour, value in enumerate(p):
        if value > p_H0:
            print(f'Predictions for {hour+1} hours ahead have same median (fail to reject H0)')
        else:
            print(f'Predictions for {hour+1} hours ahead do not have same median (reject H0)')
    return stat, p

In [18]:
shapiro_results = []
for key in all_models:
    model, lookback, depth = key.split("-")

    print(f"\nShapiro-Wilko test: Depth {depth}")
    stat, p = verify_distribution_shapiro(all_preds[key], p_H0=0.05)
    shapiro_results.append([f"Depth {i+1}", stat, p])

    shapiro_df = pd.DataFrame(
        shapiro_results,
        columns=["Depth", "statistic", "p-value"]
    )
    shapiro_df.set_index("Depth")

    filename = f"shapiro-{ansatz}-lookback-{lookback}.csv"
    print(f"\nSaving Shapiro-wilko results in {os.path.join(path_stat, filename)}")
    shapiro_df.to_csv(os.path.join(path_stat, filename))


Shapiro-Wilko test: Depth 1
Statistics=0.9301643174667383, p=1.5205795247086303e-74
Does not follow normal distribution (reject H0)

Saving Shapiro-wilko results in /home/otto.pires/dynex-poc/statistics/Ansatz-QNN-Lookback-1/shapiro-QNN-lookback-1.csv

Shapiro-Wilko test: Depth 2
Statistics=0.8473408098239781, p=1.5040607416877125e-92
Does not follow normal distribution (reject H0)

Saving Shapiro-wilko results in /home/otto.pires/dynex-poc/statistics/Ansatz-QNN-Lookback-1/shapiro-QNN-lookback-1.csv

Shapiro-Wilko test: Depth 3
Statistics=0.8026563111678868, p=6.809322969188463e-99
Does not follow normal distribution (reject H0)

Saving Shapiro-wilko results in /home/otto.pires/dynex-poc/statistics/Ansatz-QNN-Lookback-1/shapiro-QNN-lookback-1.csv

Shapiro-Wilko test: Depth 4
Statistics=0.877124889913012, p=2.3619737018965316e-87
Does not follow normal distribution (reject H0)

Saving Shapiro-wilko results in /home/otto.pires/dynex-poc/statistics/Ansatz-QNN-Lookback-1/shapiro-QNN-lookb

/tmp/ipykernel_2682985/2147716133.py:2: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 26112.
  stat, p = stats.shapiro(data)


In [19]:
kruskal_results = []

# Comparação entre todos os Depths
print("\nKruskal-Wallis analysis for all Depths")
stat, p = verify_distribution_kruskal(*all_preds.values(), p_H0=0.05)
kruskal_results.append(["All Depths", stat, p])

# Comparação Depths + valores reais
print("\nKruskal-Wallis test for all Depths + Y Dataset")
all_depths_plus_real = list(all_preds.values()) + [y_test]
stat, p = verify_distribution_kruskal(*all_depths_plus_real, p_H0=0.05)
kruskal_results.append(["All Depths + Real", stat, p])

# Criar DataFrame
kruskal_df = pd.DataFrame(kruskal_results, columns=["Comparison", "statistic", "p-value"])
kruskal_df.set_index("Comparison", inplace=True)

filename = f"kruskal-{ansatz}-lookback-{lookback}.csv"
print(f"\nSaving Kruskal-Wallis results in {os.path.join(path_stat, filename)}")
kruskal_df.to_csv(os.path.join(path_stat, filename))


Kruskal-Wallis analysis for all Depths
Statistics=[13084.06165966 16263.35871278 15849.10609907 15728.33246844
 15829.8603173  14879.61374708], p=[0. 0. 0. 0. 0. 0.]
Predictions for 1 hours ahead do not have same median (reject H0)
Predictions for 2 hours ahead do not have same median (reject H0)
Predictions for 3 hours ahead do not have same median (reject H0)
Predictions for 4 hours ahead do not have same median (reject H0)
Predictions for 5 hours ahead do not have same median (reject H0)
Predictions for 6 hours ahead do not have same median (reject H0)

Kruskal-Wallis test for all Depths + Y Dataset
Statistics=[15200.3927817  17902.12830227 17550.10047459 17447.46821573
 17533.74557909 16726.23533422], p=[0. 0. 0. 0. 0. 0.]
Predictions for 1 hours ahead do not have same median (reject H0)
Predictions for 2 hours ahead do not have same median (reject H0)
Predictions for 3 hours ahead do not have same median (reject H0)
Predictions for 4 hours ahead do not have same median (reject H0

In [20]:
wilcox_matrix = [[f"Depth {i+1}"] for i in range(max_layers)]

for i in range(max_layers):
    for ii in range(max_layers):
        if i == ii:
            wilcox_matrix[i].append(1)
        else:
            print(f"Wilcoxon test Depth {i+1} against Depth {ii+1}\n")
            stat, p = verify_distribution_wilcoxtest(all_preds[ansatz+"-"+lookback+"-"+str(i+1)][:,0]
                                                        ,all_preds[ansatz+"-"+lookback+"-"+str(ii+1)][:,0]
                                                        , 0.05)
            wilcox_matrix[i].append(p)
wilcox = pd.DataFrame(wilcox_matrix)
wilcox.columns = ["Index"]+[f"Depth {i+1}" for i in range(max_layers)]
wilcox.set_index('Index')

filename = f"wilcoxon_matrix-{ansatz}-lookback-{lookback}.csv"
print(f"\nSaving Wilcoxon matrix in {os.path.join(path_stat, filename)}")
wilcox.to_csv(os.path.join(path_stat, filename))

Wilcoxon test Depth 1 against Depth 2

Statistics=1975172.500, p=0.000
Different distribution (reject H0)
Wilcoxon test Depth 1 against Depth 3

Statistics=208968.000, p=0.000
Different distribution (reject H0)
Wilcoxon test Depth 1 against Depth 4

Statistics=30201.000, p=0.000
Different distribution (reject H0)
Wilcoxon test Depth 2 against Depth 1

Statistics=1975172.500, p=0.000
Different distribution (reject H0)
Wilcoxon test Depth 2 against Depth 3

Statistics=29.000, p=0.000
Different distribution (reject H0)
Wilcoxon test Depth 2 against Depth 4

Statistics=137424.000, p=0.000
Different distribution (reject H0)
Wilcoxon test Depth 3 against Depth 1

Statistics=208968.000, p=0.000
Different distribution (reject H0)
Wilcoxon test Depth 3 against Depth 2

Statistics=29.000, p=0.000
Different distribution (reject H0)
Wilcoxon test Depth 3 against Depth 4

Statistics=188985.000, p=0.000
Different distribution (reject H0)
Wilcoxon test Depth 4 against Depth 1

Statistics=30201.000, p